<a href="https://colab.research.google.com/github/Gaurang983/rsp-warehouse-inspection/blob/index.html/q1120205_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q pybullet matplotlib numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import json, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pybullet as p
import pybullet_data
assets = Path("warehouse_assets")
assets.mkdir(exist_ok=True)
if p.isConnected(): p.disconnect()
client = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")



0

In [ ]:

def box_model(name, size, color):
    # Static semantic objects share a simple geometry generator but distinct models.
    xyz = " ".join(map(str, size))
    rgba = " ".join(map(str, color))
    text = f'''




'''
    path = assets / f"{name}.urdf"
    path.write_text(text)
    return str(path)
models = {
    "barrier": box_model("barrier", [0.18, 2.0, 0.7], [0.6, 0.63, 0.7, 1]),
    "shelf": box_model("shelf", [0.7, 1.0, 0.9], [0.8, 0.45, 0.15, 1]),
    "inspection_zone": box_model("inspection_zone", [0.65, 0.65, 0.02], [0.15, 0.8, 0.3, 1]),
    "landmark": box_model("landmark", [0.2, 0.2, 1.0], [0.2, 0.35, 1, 1]),
}
# Two driven wheel joints and a passive support sphere.
robot_xml = '''


'''
for side, y in [("left", 0.18), ("right", -0.18)]:
    robot_xml += f'''


'''
robot_xml += '''
'''
(assets / "inspector.urdf").write_text(robot_xml)


10

In [ ]:
layout = [
    ("west_barrier", "barrier", "obstacle", [-2.4, 0, 0.35]),
    ("east_barrier", "barrier", "obstacle", [2.4, 0, 0.35]),
    ("shelf_A", "shelf", "obstacle", [-1.3, 0.8, 0.45]),
    ("shelf_B", "shelf", "obstacle", [1.3, 0.8, 0.45]),
    ("inspection_A", "inspection_zone", "target", [-1.3, -0.8, 0.011]),
    ("inspection_B", "inspection_zone", "target", [1.3, -0.8, 0.011]),
    ("home_zone", "inspection_zone", "home", [0, -1.8, 0.011]),
    ("reference_A", "landmark", "landmark", [-2.0, 1.8, 0.5]),
    ("reference_B", "landmark", "landmark", [2.0, 1.8, 0.5]),
]
semantics = {}
for name, model, role, position in layout:
    body = p.loadURDF(models[model], position, useFixedBase=True)
    semantics[body] = dict(name=name, model=model, role=role)
robot = p.loadURDF(str(assets / "inspector.urdf"), [0, -1.8, 0.155])
print(json.dumps(semantics, indent=2))

def nearest_object(role):
    robot_xy = np.array(p.getBasePositionAndOrientation(robot)[0][:2])
    candidates = [(float(np.linalg.norm(np.array(p.getBasePositionAndOrientation(body)[0][:2]) - robot_xy)), body)
                  for body, metadata in semantics.items() if metadata["role"] == role]
    if not candidates: return None
    distance, body = min(candidates)
    return {**semantics[body], "body_id": body, "distance_m": round(distance, 3)}
print("Nearest inspection target:", nearest_object("target"))


error: Cannot load URDF file.